# Case study: the cleaned decision table with a log of cleaning decisions

*Session 4, block 3 · Author: course team · Licence: CC-BY-4.0*

**Task.** Produce the cleaned table of BTI decisions that later sessions can use, and a log that records each
cleaning decision: the rule, how many rows it affected, the action and why.

The pipeline is a sequence of small functions. Each function takes a DataFrame, returns a new one and writes
an entry into the log. Nothing is changed by hand.

Output: `case-study/data/decisions_clean.parquet` and `cleaning_log.csv` next to it (the `data` folder is not
part of the repository). Set the environment variable `CLEAN_OUT` to write somewhere else.

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = Path(os.environ.get("CLEAN_OUT", DATA / "decisions_clean.parquet"))
raw = pd.read_parquet(DATA / "train.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet")
print(raw.shape)

## 1. The decision log

In [ ]:
log = []


def record(step: str, rule: str, affected: int, action: str, reason: str, n_rows: int) -> None:
    log.append({"step": step, "rule": rule, "rows_affected": affected, "action": action,
                "reason": reason, "rows_after": n_rows})

## 2. Cleaning steps

Each step follows the same pattern: find the rows, decide (remove, correct, flag, transform), record.

In [ ]:
OFFICIAL = {"AT": {"de"}, "BE": {"nl", "fr", "de"}, "BG": {"bg"}, "CY": {"el"}, "CZ": {"cs"}, "DE": {"de"},
            "DK": {"da"}, "EE": {"et"}, "ES": {"es"}, "FI": {"fi", "sv"}, "FR": {"fr"}, "GB": {"en"},
            "GR": {"el"}, "HR": {"hr"}, "HU": {"hu"}, "IE": {"en", "ga"}, "IT": {"it"}, "LT": {"lt"},
            "LU": {"fr", "de"}, "LV": {"lv"}, "MT": {"mt", "en"}, "NL": {"nl"}, "PL": {"pl"}, "PT": {"pt"},
            "RO": {"ro"}, "SE": {"sv"}, "SI": {"sl"}, "SK": {"sk"}, "XI": {"en"}}


def drop_template_descriptions(df):
    template = df["description"].str.upper().str.contains("SQL{", regex=False)
    out = df[~template].copy()
    record("1 completeness", "description is a database template (SQL{...})", int(template.sum()), "remove",
           "the text says nothing about the goods; 73 German decisions of 2017-2020", len(out))
    return out


def fix_placeholder_end_date(df):
    out = df.copy()
    placeholder = out["end_date"].dt.year.eq(1900)
    out.loc[placeholder, "end_date"] = pd.NaT
    out["annulled"] = out["invalidation_reason"].eq("55")
    record("2 validity", "end_date 1900-01-01 (placeholder of annulled decisions, code 55)", int(placeholder.sum()),
           "set to missing; add flag annulled", "a placeholder is a hidden missing value, not a date", len(out))
    return out


def normalise_description(df):
    out = df.copy()
    before = out["description"]
    after = (before.str.replace("\r\n", "\n", regex=False).str.replace("\r", "\n", regex=False)
                   .str.replace("\u00a0", " ", regex=False)              # no-break space
                   .str.replace(r"[ \t]+", " ", regex=True)               # repeated spaces and tabs
                   .str.replace(r" *\n *", "\n", regex=True)             # spaces around line breaks
                   .str.strip())
    out["description"] = after
    record("3 validity", "Windows line breaks, no-break spaces, repeated spaces in description",
           int(before.ne(after).sum()), "correct", "keeps line breaks (a feature) but makes lengths comparable", len(out))
    return out


def flag_codes(df):
    out = df.copy()
    out["heading_outdated"] = ~out["heading"].isin(nomenclature["heading"])
    out["cn_code_short"] = ~out["cn_code"].str.fullmatch(r"\d{8}")
    record("4 validity", "heading not in HS 2022 (8803, deleted in the 2022 revision)", int(out["heading_outdated"].sum()),
           "flag (keep)", "label valid when decided; a model for 2024+ may exclude these rows", len(out))
    record("5 validity", "CN code with only 4 or 6 digits", int(out["cn_code_short"].sum()), "flag (keep)",
           "the heading (first four digits) is still known; the task needs only the heading", len(out))
    return out


def flag_duplicate_descriptions(df):
    out = df.copy()
    norm = out["description"].str.lower().str.replace(r"\s+", " ", regex=True)
    out["is_duplicate_description"] = norm.duplicated(keep=False)
    out["n_headings_same_description"] = out.groupby(norm)["heading"].transform("nunique")
    record("6 uniqueness", "description identical to another decision (ignoring case and spaces)",
           int(out["is_duplicate_description"].sum()), "flag (keep)",
           "renewals are real decisions; but they must not cross a train/validation split (Session 7)", len(out))
    return out


def flag_unexpected_language(df):
    out = df.copy()
    official = [lang in OFFICIAL[c] for c, lang in zip(out["issuing_country"], out["language"])]
    out["language_unexpected"] = ~pd.Series(official, index=out.index) & out["language"].ne("en")
    record("7 consistency", "language neither official in the issuing country nor English",
           int(out["language_unexpected"].sum()), "flag (keep)", "probably a wrong language code; check before language-specific processing", len(out))
    return out


def add_missing_indicators(df):
    out = df.copy()
    out["keywords_missing"] = out["keywords"].isna()
    out["ended_early"] = out["invalidation_reason"].notna()
    record("8 completeness", "keywords missing", int(out["keywords_missing"].sum()), "keep NULL, add indicator",
           "not MCAR (depends on country and year, workbook 08); keywords are not a model input anyway", len(out))
    record("9 completeness", "invalidation_reason missing", int((~out["ended_early"]).sum()),
           "keep NULL, add ended_early", "structural: valid or normally expired decisions have no reason", len(out))
    return out

In [ ]:
df = (raw.pipe(drop_template_descriptions).pipe(fix_placeholder_end_date).pipe(normalise_description)
         .pipe(flag_codes).pipe(flag_duplicate_descriptions).pipe(flag_unexpected_language)
         .pipe(add_missing_indicators))
pd.DataFrame(log)

## 3. Derived columns and transformations

Simple numeric features of the description (the same as in the Session 2 workspace) and the validity
duration. The length is strongly right-skewed. We keep the raw columns and add transformed ones. A
**Box–Cox** transformation estimates a power λ that makes the distribution as close to normal as possible
(λ = 0 is the logarithm); it requires positive values. Counts with zeros (digits) get `log1p`.

In [ ]:
from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer

text = df["description"]
letters = text.str.count(r"[^\W\d_]")
df["n_chars"] = text.str.len()
df["n_digits"] = text.str.count(r"[0-9]")
df["n_lines"] = text.str.count("\n") + 1
df["upper_share"] = (text.str.count(r"[^\W\d_a-zß-ÿ]") / letters).where(letters > 0, 0.0)
df["validity_days"] = (df["end_date"] - df["start_date"]).dt.days

bc = PowerTransformer(method="box-cox", standardize=False).fit(df[["n_chars"]])
df["n_chars_boxcox"] = bc.transform(df[["n_chars"]]).ravel()
df["log1p_digits"] = np.log1p(df["n_digits"])
df["log1p_lines"] = np.log1p(df["n_lines"])

print(f"Box-Cox lambda for n_chars: {bc.lambdas_[0]:.3f}")
pd.DataFrame({
    "n_chars": [skew(df["n_chars"]), skew(np.log(df["n_chars"])), skew(df["n_chars_boxcox"])],
    "n_digits": [skew(df["n_digits"]), skew(df["log1p_digits"]), np.nan],
}, index=["raw", "log / log1p", "Box-Cox"]).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), layout="constrained")
for ax, col, title in zip(axes, ["n_chars", "n_chars_boxcox", "log1p_digits"],
                          ["description length (characters)", "length, Box-Cox", "log(1 + digits)"]):
    ax.hist(df[col], bins=60, color="#2a78d6")
    ax.set(title=title)
    ax.spines[["top", "right"]].set_visible(False)
plt.show()
record("10 transformation", "n_chars and n_digits strongly right-skewed", len(df), "add transformed columns",
       f"Box-Cox (lambda = {bc.lambdas_[0]:.2f}) and log1p; raw columns kept", len(df))

## 4. Outliers: flag, do not delete

**Univariate.** The MAD rule on the Box–Cox scale flags unusually short or long descriptions.

**Multivariate.** The Mahalanobis distance on two description features (Box–Cox length and log digits)
flags unusual *combinations*, for example a long text without any number, or a short text that consists
mostly of numbers. We estimate centre and covariance robustly with the minimum covariance determinant (MCD)
on a random sample of 20,000 decisions and compare the squared distance with the 99.9 % quantile of the
chi-square distribution with 2 degrees of freedom.

In [ ]:
from scipy.stats import chi2
from sklearn.covariance import MinCovDet

x = df["n_chars_boxcox"]
mad = (x - x.median()).abs().median()
df["outlier_length_mad"] = (0.6745 * (x - x.median()) / mad).abs() > 3.5
record("11 outliers (univariate)", "robust z-score of Box-Cox description length > 3.5", int(df["outlier_length_mad"].sum()),
       "flag (keep)", "very short texts (e.g. 'TEST') are suspicious, very long ones genuine; inspect", len(df))

cols = ["n_chars_boxcox", "log1p_digits"]
Z = df[cols].to_numpy()
sample = Z[np.random.default_rng(2026).choice(len(Z), 20_000, replace=False)]
mcd = MinCovDet(random_state=0).fit(sample)
df["mahalanobis_d2"] = mcd.mahalanobis(Z)
df["outlier_mahalanobis"] = df["mahalanobis_d2"] > chi2.ppf(0.999, df=len(cols))
record("12 outliers (multivariate)", "robust Mahalanobis d2 of (Box-Cox length, log digits) > chi2(2) 99.9 %",
       int(df["outlier_mahalanobis"].sum()), "flag (keep)", "unusual form of the text; inspect before use", len(df))
print(f"flagged: {df['outlier_mahalanobis'].mean():.1%}")
df.loc[df["outlier_mahalanobis"], ["language", "n_chars", "n_digits", "n_lines", "upper_share", "description"]].sample(6, random_state=1)

Look at the flagged rows before you trust the flag. The Mahalanobis rule assumes one elliptical cloud.
When the course team first used four features (adding log lines and the share of upper-case letters), the
robust distance flagged 23.7 % of all decisions: the share of upper-case letters is **bimodal**, because about
7.6 % of the descriptions are written entirely or mostly in capitals. MCD fits the majority (normal case) and
then flags the whole minority group. The rule found a *kind* of description, not errors. Check the
distribution of each variable first (histogram), and use the model-based methods of Session 11 for data with
several groups.

**TODO 0.** Run the multivariate step with `cols = ["n_chars_boxcox", "log1p_digits", "log1p_lines",
"upper_share"]` and `df.groupby(df["upper_share"] > 0.9)["outlier_mahalanobis"].mean()`. What do you see?

## 5. Validate the result, then save it

In [ ]:
assert df["bti_reference"].is_unique
assert not df["description"].str.upper().str.contains("SQL{", regex=False).any()
assert not df["end_date"].dt.year.eq(1900).any()
known_end = df["end_date"].notna()
assert (df.loc[known_end, "end_date"] >= df.loc[known_end, "start_date"]).all()
assert not df["description"].str.contains("\r", regex=False).any()
assert df["description"].str.len().gt(0).all()
assert len(df) == len(raw) - log[0]["rows_affected"]
print("all assertions passed:", df.shape)

In [ ]:
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT, index=False)
log_df = pd.DataFrame(log)
log_df.to_csv(OUT.with_name("cleaning_log.csv"), index=False)
print(f"wrote {OUT} ({OUT.stat().st_size / 1e6:.0f} MB) and {OUT.with_name('cleaning_log.csv').name}")
log_df

**TODO.** (1) Add a step of your own, for example flagging descriptions that are only a test entry
("TEST", "prova"), and record it. (2) Run the tests in [`quality/test_bti_quality.py`](quality/test_bti_quality.py);
the tests on the cleaned table should now pass. (3) Write three sentences for the data card that summarise
the cleaning.